# 07 - Leakage Field Classification & EDA Summary



## Setup


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

DATA_PATH = Path('../../data/raw data/DataCoSupplyChainDataset.csv')  

try:
    df = pd.read_csv(DATA_PATH, encoding='utf-8')
except UnicodeDecodeError:
    print("UTF-8 failed, falling back to ISO-8859-1")
    df = pd.read_csv(DATA_PATH, encoding='ISO-8859-1')

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


UTF-8 failed, falling back to ISO-8859-1
Shape: 180,519 rows x 53 columns


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


## Leakage field classification

Every column marked as **safe** (order time), **excluded** (post outcome), or **target**.

In [2]:
leakage_classification = {
    # column name: 'safe' | 'excluded' | 'target'
    'Delivery Status': 'excluded',
    'Late_delivery_risk': 'target',
    'Days for shipping (real)': 'excluded',
    'shipping date (DateOrders)': 'excluded',
    'Shipping Mode': 'safe',
    'Order Region': 'safe',
    'Order Country': 'safe',
    'Order City': 'safe',
    'Order State': 'safe',
    'Category Name': 'safe',
    'Category Id': 'safe',
    'Order Item Quantity': 'safe',
    'Sales': 'safe',
    'Benefit per order': 'safe',  # need to check
    'Customer Segment': 'safe',
    'Type': 'safe',
    'Days for shipment (scheduled)': 'safe',
}

classification_df = pd.DataFrame(list(leakage_classification.items()), columns=['column', 'classification'])
classification_df


,column,classification
0,Delivery Status,excluded
1,Late_delivery_risk,target
2,Days for shipping (real),excluded
3,shipping date (DateOrders),excluded
4,Shipping Mode,safe
5,Order Region,safe
6,Order Country,safe
7,Order City,safe
8,Order State,safe
9,Category Name,safe


**Note on `Benefit per order` / `Sales`:** useful for the secondary lens (prioritisation), but need to think whether using them as classification model *inputs* risks a subtle leakage adjacent issue. Log the discussion even if the conclusion is "safe to use."

The worry wasn't that these columns are "impossible to know" at order time - they're not. Sales and profit per order are calculated the moment someone places an order (price × quantity, minus cost), so there's no timing problem, unlike Delivery Status which literally can't exist until the order finishes shipping.

The worry was something subtler: could the model use these numbers to "cheat" by learning a shortcut that has nothing to do with real delivery risk?

Here's the kind of thing that raises that flag: imagine, just hypothetically, that expensive orders happened to get shipped more carefully (better packaging, priority handling) and therefore were less likely to be late - not because of anything about when they were placed, but because of some operational quirk in how the business handles high value orders. If that pattern existed, the model might learn "high Sales number → predict not late" - and that prediction would work on your test set, but it wouldn't really be learning about delivery risk. It'd be learning a proxy that happens to correlate, which is a shakier, less trustworthy kind of signal than something like Shipping Mode, which has an obvious, direct causal reason to affect lateness.

**Resolution, based on everything found across Notebooks 01–06:**

These fields are **safe to use as classification model inputs** - they're genuinely known at order time (a customer's order total and expected margin are known the moment the order is placed, well before fulfillment), so there's no *temporal* leakage here. The earlier caution was worth raising, but on reflection the concern doesn't hold up: nothing about `Sales` or `Benefit per order` depends on knowing the delivery outcome.

**The real issue we found with `Benefit per order` isn't leakage - it's data quality (see Notebook 06):** 10.49% of rows are outliers by IQR, including a minimum of –$4,274.98. That's a *preprocessing* concern (capping/winsorizing for model stability), not a leakage concern, and it's a separate decision from whether the column is safe to use at all.

**Where `Sales`/`Benefit per order` genuinely need careful, separate design thought is the secondary lens's priority formula** - specifically how a large negative profit order should factor into a `risk × value` priority score. That's a business logic decision, not a leakage question, and it's tracked separately in Notebook 06's findings.

**Conclusion for the classification table above:** both columns remain classified as `safe`. No change needed.


## EDA summary - key findings

| # | Finding | Implication for next stages |
|---|---|---|
| 1 | **Unit of analysis:** data is order-line level — 180,519 rows collapse to 65,752 unique orders (avg. 2.75 lines/order). Target (`Late_delivery_risk`) is consistent across all line items within an order, but numeric fields (Sales, quantity, price) vary across lines. | Aggregate to order level in Stage 3 before modelling. Define per-column aggregation rule (sum for value fields, first/mode for order-level categoricals, first for target) and log it. |
| 2 | **Target audit:** `Late_delivery_risk` maps cleanly and consistently to `Delivery Status`, with zero inconsistencies. One judgement call surfaced: `Shipping canceled` (7,754 orders, 4.3%) is coded as 0 (not late), bundled with genuinely on-time deliveries. | Target is trustworthy — audit confirms it. Group decision needed and logged: keep canceled-as-0 as-is (recommended, now evidence-backed), or handle separately. |
| 3 | **Class split:** 54.83% late / 45.17% not-late — close to balanced, confirmed on our own extracted data. | No aggressive imbalance-handling (SMOTE, heavy weighting) needed. Recall still prioritised in Stage 7, but for cost-sensitivity reasons (missed late delivery costs more than a false alarm), not imbalance reasons — important distinction for the viva. |
| 4 | **Temporal drift:** late rate stable across 2015–2017 (54.5–55.1%), no meaningful drift. But the dataset only extends to January 2018 — a thin, single-month, 2,123-row tail. Day-of-week and month patterns are essentially flat (no strong calendar effect). | A naive calendar-year chronological split (train 2015–17, test 2018) would give a too-small, seasonally-narrow test set. Use a proportion-based chronological split instead (last ~15–20% of orders by date) — keeps "test on the future" logic without the thin-sample problem. Deprioritise calendar-based feature engineering (weak standalone signal). |
| 5 | **Geographic:** region-level spread is narrow (48.8–58.0%, ~9pp); country-level spread is much wider (36.9–67.0%, ~30pp), though some standout countries have smaller sample sizes. Data-quality issue found: `Order Country` mixes English and Spanish names inconsistently. | Country-level delay rate is a stronger feature candidate than region — consider target-encoding at country level. Standardise country naming in Stage 3 before encoding. Responsible-AI flag: highest-risk regions (Central/South Asia, East/Central Africa) should inform proactive support, not deprioritisation, in the final recommendation. |
| 6 | **Shipping Mode — by far the strongest signal found:** First Class 95.3% late vs. Standard Class 38.1% late, a 57pp spread (vs. ~30pp for country, ~9pp for region). Likely driven by tighter scheduled-delivery windows for faster modes. Product Category shows a moderate, more typical 21pp spread. | Shipping Mode will likely dominate every model — validates the rule-based baseline as a genuinely tough benchmark. Check for multicollinearity between Shipping Mode and `Days for shipment (scheduled)` in Stage 3/4 before finalising the feature set. |
| 7 | **Numeric/outliers:** `Sales` and the two discrete fields (quantity, scheduled days) are clean, negligible outliers. `Benefit per order` has 10.49% outliers by IQR, including a minimum of –$4,274.98. | Cap/winsorize `Benefit per order` for model training stability in Stage 3. Separately, explicitly design how negative-profit orders factor into the secondary lens's priority formula — this is a business-logic decision distinct from general data cleaning. |
| 8 | **Leakage classification finalised:** `Delivery Status`, `Late_delivery_risk` (target), `Days for shipping (real)`, and `shipping date (DateOrders)` are the only excluded/target fields; all other planned features are confirmed safe order-time inputs, including `Sales`/`Benefit per order` (resolved above — data quality issue, not leakage). | This table is the authoritative reference for Stage 3/4 feature selection — copy it into `docs/data_dictionary.md`. |

### next steps

1. **Log five decisions in `DECISION_LOG.md`** before Stage 3 begins:
   - Aggregation rule per column for collapsing line items to order level (finding #1)
   - How `Shipping canceled` orders are handled in the target (finding #2)
   - Final split strategy: proportion based chronological split, ~15–20% most recent orders as test (finding #4)
   - Country name standardisation approach (finding #5)
   - `Benefit per order` capping strategy + negative profit handling in the priority formula (finding #7)
2. **Copy the finalised leakage classification table into `docs/data_dictionary.md`** as the single source of truth for what Stage 3/4 can and cannot use as model inputs.
3. **Move to Stage 3: Data Preprocessing** - aggregation, missing value handling (drop `Product Description`, check `Order Zipcode` vs. country), categorical standardisation, outlier capping, and the chronological train/test split, in that order.
